# Module 05 — Evaluation

In this notebook you will learn:
- Why evaluation is critical for maintaining LLM quality
- How to measure retrieval quality (RAG)
- How to measure generation quality
- How to detect model degradation over time


## 1. What Are We Measuring?

A RAG system has two components to evaluate:

| Component | Question | Key Metrics |
|-----------|----------|-------------|
| Retrieval | Did we find the right documents? | Precision@K, Recall@K, MRR |
| Generation | Did we generate a good answer? | ROUGE, BERTScore, faithfulness |

The most important metric in production: **does the user get the right answer?**


## 2. Building an Evaluation Dataset

You need question-answer pairs to evaluate against. Golden answers give you a ground truth.


In [ ]:
# A small evaluation set — in practice you'd have hundreds of these
eval_dataset = [
    {
        'question': 'What does RAG stand for?',
        'golden_answer': 'Retrieval-Augmented Generation',
        'relevant_doc_ids': ['doc_0']
    },
    {
        'question': 'How does LoRA reduce training cost?',
        'golden_answer': 'LoRA adds small low-rank matrices instead of updating all weights',
        'relevant_doc_ids': ['doc_1']
    },
    {
        'question': 'What is the purpose of embeddings?',
        'golden_answer': 'Embeddings are vector representations that capture semantic meaning',
        'relevant_doc_ids': ['doc_3']
    },
]

print(f'Evaluation set: {len(eval_dataset)} questions')

## 3. Retrieval Evaluation — Precision@K and Recall@K


In [ ]:
import chromadb

# Rebuild the knowledge base from Module 03
client = chromadb.Client()
collection = client.create_collection('eval_kb')

documents = [
    'RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model.',
    'LoRA is a parameter-efficient fine-tuning technique that adds low-rank matrices to frozen model weights.',
    'ChromaDB is an open-source vector database designed for AI applications.',
    'Embeddings are dense vector representations of text that capture semantic meaning.',
    'The context window of an LLM determines how much text it can process in one pass.',
]

collection.add(
    documents=documents,
    ids=[f'doc_{i}' for i in range(len(documents))]
)

def precision_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    retrieved_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    hits = sum(1 for doc_id in retrieved_k if doc_id in relevant_set)
    return hits / k

def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    retrieved_k = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    hits = len(retrieved_k & relevant_set)
    return hits / len(relevant_set)

# Run retrieval evaluation
K = 3
precision_scores = []
recall_scores = []

for item in eval_dataset:
    results = collection.query(query_texts=[item['question']], n_results=K)
    retrieved_ids = results['ids'][0]
    
    p = precision_at_k(retrieved_ids, item['relevant_doc_ids'], K)
    r = recall_at_k(retrieved_ids, item['relevant_doc_ids'], K)
    
    precision_scores.append(p)
    recall_scores.append(r)
    
    print(f"Q: {item['question'][:50]}")
    print(f"   Retrieved: {retrieved_ids} | Precision@{K}: {p:.2f} | Recall@{K}: {r:.2f}\n")

print(f'Mean Precision@{K}: {sum(precision_scores)/len(precision_scores):.3f}')
print(f'Mean Recall@{K}:    {sum(recall_scores)/len(recall_scores):.3f}')

## 4. Generation Evaluation — ROUGE Score

ROUGE measures overlap between the generated answer and the golden answer.
It's not perfect, but it's a useful automated signal.


In [ ]:
from evaluate import load

rouge = load('rouge')

# Simulated model outputs (in practice these come from your RAG pipeline)
predictions = [
    'RAG stands for Retrieval-Augmented Generation, combining retrieval with language models.',
    'LoRA reduces cost by adding small matrices instead of training all parameters.',
    'Embeddings capture semantic meaning as dense numerical vectors.',
]

references = [item['golden_answer'] for item in eval_dataset]

results = rouge.compute(predictions=predictions, references=references)

print('ROUGE Scores:')
for metric, score in results.items():
    print(f'  {metric}: {score:.3f}')

## 5. Tracking Quality Over Time

Run evaluations on a schedule and log scores to detect degradation.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

METRICS_FILE = Path('./eval_history.jsonl')

def log_evaluation(precision: float, recall: float, rouge_l: float):
    """Append evaluation results to a log file for trend tracking."""
    record = {
        'timestamp': datetime.now().isoformat(),
        'precision_at_3': precision,
        'recall_at_3': recall,
        'rouge_l': rouge_l
    }
    with open(METRICS_FILE, 'a') as f:
        f.write(json.dumps(record) + '\n')
    print(f'Logged: {record}')

def check_for_degradation(threshold: float = 0.1):
    """Alert if recent scores have dropped significantly vs baseline."""
    if not METRICS_FILE.exists():
        return
    
    records = [json.loads(line) for line in METRICS_FILE.read_text().strip().split('\n')]
    
    if len(records) < 2:
        return
    
    baseline = records[0]['rouge_l']
    latest = records[-1]['rouge_l']
    drop = baseline - latest
    
    if drop > threshold:
        print(f'WARNING: ROUGE-L dropped by {drop:.3f} from baseline {baseline:.3f} to {latest:.3f}')
    else:
        print(f'OK: ROUGE-L stable ({latest:.3f}, baseline {baseline:.3f})')

# Log current run
log_evaluation(
    precision=sum(precision_scores)/len(precision_scores),
    recall=sum(recall_scores)/len(recall_scores),
    rouge_l=results['rougeL']
)

check_for_degradation()

## Key Takeaways

1. **Evaluate both retrieval and generation** — a bad retriever will make even a perfect model fail
2. **Build an eval dataset early** — golden answers are worth the effort
3. **Log metrics over time** — you can't detect degradation without a history
4. **Set alert thresholds** — automate monitoring so you catch regressions quickly
5. **ROUGE is a proxy** — complement with human evaluation for critical systems

## Congratulations!
You've completed the core learning path. You now know how to:
- Load and run LLMs
- Fine-tune with LoRA
- Build a RAG pipeline
- Keep knowledge fresh
- Evaluate and monitor quality
